In [1]:
import os

os.chdir("..")
from src.ingestion.database import get_engine

engine = get_engine()

In [4]:
import pandas as pd

validation_queries = {

    "Total fact rows": """
        SELECT COUNT(*)
        FROM fact_sales
    """,

    "Duplicate order items": """
        SELECT COUNT(*)
        FROM (
            SELECT order_id, order_item_id
            FROM fact_sales
            GROUP BY order_id, order_item_id
            HAVING COUNT(*) > 1
        ) x
    """,

    "Missing customer keys": """
        SELECT COUNT(*)
        FROM fact_sales
        WHERE customer_key IS NULL
    """,

    "Missing product keys": """
        SELECT COUNT(*)
        FROM fact_sales
        WHERE product_key IS NULL
    """,

    "Missing seller keys": """
        SELECT COUNT(*)
        FROM fact_sales
        WHERE seller_key IS NULL
    """,

    "Missing date keys": """
        SELECT COUNT(*)
        FROM fact_sales
        WHERE purchase_date_key IS NULL
    """,

    "Incorrect revenue calculations": """
        SELECT COUNT(*)
        FROM fact_sales
        WHERE ABS(
            total_item_value - (price + freight_value)
        ) > 0.01
    """,

    "Incorrect delivery calculations": """
        SELECT COUNT(*)
        FROM fact_sales
        WHERE delivered_date IS NOT NULL
        AND delivery_days != DATEDIFF(
            delivered_date,
            purchase_timestamp
        )
    """
}


validation_results = []
from sqlalchemy import text

with engine.connect() as connection:

    for check_name, query in validation_queries.items():

        result = connection.execute(
            text(query)
        ).scalar()

        validation_results.append({
            "validation": check_name,
            "result": result,
            "status": "PASS"
                if (
                    check_name == "Total fact rows"
                    or result == 0
                )
                else "FAIL"
        })


validation_df = pd.DataFrame(validation_results)

validation_df

,validation,result,status
0,Total fact rows,112650,PASS
1,Duplicate order items,0,PASS
2,Missing customer keys,0,PASS
3,Missing product keys,0,PASS
4,Missing seller keys,0,PASS
5,Missing date keys,0,PASS
6,Incorrect revenue calculations,0,PASS
7,Incorrect delivery calculations,0,PASS


###     Fact Payments

In [5]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(order_id, '-', payment_sequential))
        AS unique_payments,
    COUNT(DISTINCT order_id)
        AS unique_orders,
    SUM(payment_value) AS total_payment_value
FROM fact_payments;
"""

payment_summary = pd.read_sql(query, engine)

payment_summary

,total_rows,unique_payments,unique_orders,total_payment_value
0,103886,103886,99440,16008872.12


## FACT REVIEWS

In [6]:
query = """
SELECT
    COUNT(*) AS total_reviews,

    COUNT(DISTINCT review_id) AS unique_review_ids,

    COUNT(DISTINCT order_id) AS reviewed_orders,

    ROUND(AVG(review_score), 2) AS average_score,

    SUM(review_score = 5) AS five_star_reviews,

    SUM(review_score = 1) AS one_star_reviews

FROM fact_reviews;
"""

review_summary = pd.read_sql(query, engine)

review_summary

,total_reviews,unique_review_ids,reviewed_orders,average_score,five_star_reviews,one_star_reviews
0,99224,98410,98673,4.09,57328.0,11424.0


In [7]:
query = """
SELECT
    review_score,
    COUNT(*) AS review_count,
    ROUND(
        100 * COUNT(*) /
        SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM fact_reviews
GROUP BY review_score
ORDER BY review_score;
"""

review_distribution = pd.read_sql(query, engine)

review_distribution

,review_score,review_count,percentage
0,1,11424,11.51
1,2,3151,3.18
2,3,8179,8.24
3,4,19142,19.29
4,5,57328,57.78


In [8]:
query = """
SELECT
    review_id,
    order_id,
    COUNT(*) AS duplicate_count
FROM fact_reviews
GROUP BY
    review_id,
    order_id
HAVING COUNT(*) > 1;
"""

review_duplicates = pd.read_sql(query, engine)

print(f"Duplicate review/order combinations: {len(review_duplicates)}")

review_duplicates

Duplicate review/order combinations: 0


,review_id,order_id,duplicate_count


### got missing values need to resolve

In [9]:
query = """
SELECT
    COUNT(*) AS missing_dates,
    MIN(review_creation_date) AS earliest_review,
    MAX(review_creation_date) AS latest_review
FROM fact_reviews
WHERE review_date_key IS NULL;
"""

pd.read_sql(query, engine)

,missing_dates,earliest_review,latest_review
0,309,2016-10-15,2017-01-04


In [10]:
query = """
SELECT
    review_creation_date,
    COUNT(*) AS review_count
FROM fact_reviews
WHERE review_date_key IS NULL
GROUP BY review_creation_date
ORDER BY review_creation_date
LIMIT 20;
"""

pd.read_sql(query, engine)

,review_creation_date,review_count
0,2016-10-15 00:00:00,2
1,2016-10-16 01:00:00,1
2,2016-10-18 00:00:00,12
3,2016-10-19 00:00:00,20
4,2016-10-20 00:00:00,17
5,2016-10-21 00:00:00,10
6,2016-10-23 00:00:00,1
7,2016-10-24 00:00:00,1
8,2016-10-25 00:00:00,22
9,2016-10-26 00:00:00,19


In [11]:
query = """
SELECT
    MIN(full_date) AS min_date,
    MAX(full_date) AS max_date,
    COUNT(*) AS total_dates
FROM dim_date;
"""

pd.read_sql(query, engine)

,min_date,max_date,total_dates
0,2016-09-04,2018-10-17,634


In [12]:
query = """
SELECT
    MIN(review_creation_date) AS min_review_date,
    MAX(review_creation_date) AS max_review_date
FROM stg_reviews;
"""

pd.read_sql(query, engine)

,min_review_date,max_review_date
0,2016-10-02,2018-08-31


### Chanes to the code for missing values

In [14]:
from sqlalchemy import text

with engine.begin() as connection:

    # 1. Create temporary calendar table
    connection.execute(text("""
        CREATE TEMPORARY TABLE temp_calendar (
            full_date DATE PRIMARY KEY
        )
    """))

    # 2. Generate dates using a numbers sequence
    connection.execute(text("""
        INSERT INTO temp_calendar (full_date)
        SELECT DATE_ADD(
            '2016-09-04',
            INTERVAL seq DAY
        )
        FROM (
            SELECT
                ones.n
                + tens.n * 10
                + hundreds.n * 100 AS seq
            FROM
                (SELECT 0 n UNION ALL SELECT 1 UNION ALL SELECT 2
                 UNION ALL SELECT 3 UNION ALL SELECT 4
                 UNION ALL SELECT 5 UNION ALL SELECT 6
                 UNION ALL SELECT 7 UNION ALL SELECT 8
                 UNION ALL SELECT 9) ones

            CROSS JOIN
                (SELECT 0 n UNION ALL SELECT 1 UNION ALL SELECT 2
                 UNION ALL SELECT 3 UNION ALL SELECT 4 UNION ALL SELECT 5
                 UNION ALL SELECT 6 UNION ALL SELECT 7 UNION ALL SELECT 8
                 UNION ALL SELECT 9) tens

            CROSS JOIN
                (SELECT 0 n UNION ALL SELECT 1 UNION ALL SELECT 2
                 UNION ALL SELECT 3 UNION ALL SELECT 4 UNION ALL SELECT 5
                 UNION ALL SELECT 6 UNION ALL SELECT 7 UNION ALL SELECT 8
                 UNION ALL SELECT 9) hundreds
        ) numbers
        WHERE DATE_ADD(
            '2016-09-04',
            INTERVAL seq DAY
        ) <= '2018-10-17'
    """))

    # 3. Insert missing dates into dim_date
    result = connection.execute(text("""
        INSERT IGNORE INTO dim_date (
            date_key,
            full_date,
            year,
            quarter,
            month,
            month_name,
            week_of_year,
            day_of_month,
            day_of_week,
            day_name,
            is_weekend
        )
        SELECT
            CAST(DATE_FORMAT(full_date, '%Y%m%d') AS UNSIGNED),
            full_date,
            YEAR(full_date),
            QUARTER(full_date),
            MONTH(full_date),
            MONTHNAME(full_date),
            WEEK(full_date, 3),
            DAY(full_date),
            DAYOFWEEK(full_date),
            DAYNAME(full_date),

            CASE
                WHEN DAYOFWEEK(full_date) IN (1, 7)
                THEN TRUE
                ELSE FALSE
            END

        FROM temp_calendar
    """))

    print(f"✓ Calendar dates inserted: {result.rowcount:,}")

✓ Calendar dates inserted: 140


In [15]:
query = """
SELECT
    MIN(full_date) AS min_date,
    MAX(full_date) AS max_date,
    COUNT(*) AS total_dates
FROM dim_date;
"""

pd.read_sql(query, engine)

,min_date,max_date,total_dates
0,2016-09-04,2018-10-17,774


In [16]:
from sqlalchemy import text

query = text("""
UPDATE fact_reviews fr
JOIN dim_date d
    ON DATE(fr.review_creation_date) = d.full_date
SET fr.review_date_key = d.date_key
WHERE fr.review_date_key IS NULL;
""")

with engine.begin() as connection:
    result = connection.execute(query)

print(f"✓ Review date keys updated: {result.rowcount:,}")

✓ Review date keys updated: 309


In [17]:
query = """
SELECT
    COUNT(*) AS missing_review_dates
FROM fact_reviews
WHERE review_date_key IS NULL;
"""

pd.read_sql(query, engine)

,missing_review_dates
0,0


final verification


In [18]:
query = """
SELECT
    review_id,
    order_id,
    COUNT(*) AS duplicate_count
FROM fact_reviews
GROUP BY
    review_id,
    order_id
HAVING COUNT(*) > 1;
"""

review_duplicates = pd.read_sql(query, engine)

print(
    f"Duplicate (review_id, order_id) combinations: "
    f"{len(review_duplicates)}"
)

review_duplicates

Duplicate (review_id, order_id) combinations: 0


,review_id,order_id,duplicate_count


# Fact schedma pass 

In [19]:
checks = {

    "fact_sales_row_count": """
        SELECT COUNT(*) = (
            SELECT COUNT(*)
            FROM stg_order_items
        )
        FROM fact_sales
    """,

    "fact_sales_duplicates": """
        SELECT COUNT(*) = 0
        FROM (
            SELECT order_id, order_item_id
            FROM fact_sales
            GROUP BY order_id, order_item_id
            HAVING COUNT(*) > 1
        ) x
    """,

    "fact_sales_revenue": """
        SELECT ABS(
            (SELECT SUM(total_item_value)
             FROM fact_sales)
            -
            (SELECT SUM(price + freight_value)
             FROM stg_order_items)
        ) < 0.01
    """,

    "fact_payment_row_count": """
        SELECT COUNT(*) = (
            SELECT COUNT(*)
            FROM stg_payments
        )
        FROM fact_payments
    """,

    "fact_payment_duplicates": """
        SELECT COUNT(*) = 0
        FROM (
            SELECT order_id, payment_sequential
            FROM fact_payments
            GROUP BY order_id, payment_sequential
            HAVING COUNT(*) > 1
        ) x
    """,

    "fact_payment_value": """
        SELECT ABS(
            (SELECT SUM(payment_value)
             FROM fact_payments)
            -
            (SELECT SUM(payment_value)
             FROM stg_payments)
        ) < 0.01
    """,

    "fact_review_row_count": """
        SELECT COUNT(*) = (
            SELECT COUNT(*)
            FROM stg_reviews
        )
        FROM fact_reviews
    """,

    "fact_review_duplicates": """
        SELECT COUNT(*) = 0
        FROM (
            SELECT review_id, order_id
            FROM fact_reviews
            GROUP BY review_id, order_id
            HAVING COUNT(*) > 1
        ) x
    """,

    "fact_review_invalid_scores": """
        SELECT COUNT(*) = 0
        FROM fact_reviews
        WHERE review_score NOT BETWEEN 1 AND 5
    """,

    "fact_review_missing_dates": """
        SELECT COUNT(*) = 0
        FROM fact_reviews
        WHERE review_date_key IS NULL
    """
}


results = []

with engine.connect() as connection:

    for check_name, query in checks.items():

        result = connection.execute(
            text(query)
        ).scalar()

        results.append({
            "check": check_name,
            "result": bool(result),
            "status": "PASS" if result else "FAIL"
        })


star_schema_validation = pd.DataFrame(results)

star_schema_validation

,check,result,status
0,fact_sales_row_count,True,PASS
1,fact_sales_duplicates,True,PASS
2,fact_sales_revenue,True,PASS
3,fact_payment_row_count,True,PASS
4,fact_payment_duplicates,True,PASS
5,fact_payment_value,True,PASS
6,fact_review_row_count,True,PASS
7,fact_review_duplicates,True,PASS
8,fact_review_invalid_scores,True,PASS
9,fact_review_missing_dates,True,PASS
